# CFNN Phase 3 — Visualization & Theory

Publication-quality visualizations and mathematical foundations across **three datasets**
of increasing complexity:

| Dataset | Dims | Task | Why |
|---------|------|------|-----|
| **Moons** | 2 | Binary classification | Interpretable 2D baseline |
| **Wine** | 13 | 3-class classification | Correlated chemical features — CFNN's natural domain |
| **California Housing** | 8 | Regression | Real-world, 20K samples, tests scalability |

For each dataset we train both **CFNN-A** (absorption) and **CFNN-D** (distillation),
then compare how the "column" behaves differently.

**Author:** Daniel Regalado Cardoso | University of Miami

## 1. Setup

In [ ]:
!git clone -b main https://github.com/DanielRegaladoUMiami/counterflow-nn.git
%cd counterflow-nn
!pip install scikit-learn matplotlib pandas pytest torchvision -q

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from src.network import CounterFlowNetwork
from src.distillation import DistillationNetwork
from src.utils import load_synthetic_dataset, load_uci_dataset, prepare_data, train_model
from src.visualization import (
    mccabe_thiele_plot, concentration_profile, driving_force_profile,
    transfer_heatmap, diagnostic_dashboard, column_schematic,
)
from src.diagnostics import print_diagnostics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')

## 2. Helper: Train CFNN-A and CFNN-D on any dataset

We define a helper that trains both architectures and returns everything
needed for visualization.

In [ ]:
def train_both_models(
    X, y, dataset_name, task='classification',
    d_gas=32, d_liquid=32, n_plates=8, n_epochs=150,
):
    """Train CFNN-A and CFNN-D, return models + scaled data."""
    print(f'\n{"="*60}')
    print(f'  Dataset: {dataset_name}')
    print(f'  Shape: {X.shape} | Task: {task}')
    print(f'{"="*60}')

    train_ld, test_ld, d_in, n_cls = prepare_data(X, y, seed=42)
    X_scaled = torch.FloatTensor(StandardScaler().fit_transform(X))

    # CFNN-A
    print(f'\n--- CFNN-A (Absorption, {n_plates} plates) ---')
    model_a = CounterFlowNetwork(
        d_in=d_in, d_gas=d_gas, d_liquid=d_liquid,
        n_plates=n_plates, d_out=n_cls, n_sweeps=2
    )
    hist_a = train_model(
        model_a, train_ld, test_ld, n_epochs=n_epochs,
        device=device, task=task, verbose=True, print_every=50
    )
    model_a.to('cpu')

    # CFNN-D
    n_rect = n_plates // 2
    n_strip = n_plates - n_rect
    print(f'\n--- CFNN-D (Distillation, {n_rect}+{n_strip} plates) ---')
    model_d = DistillationNetwork(
        d_in=d_in, d_gas=d_gas, d_liquid=d_liquid,
        n_plates_rect=n_rect, n_plates_strip=n_strip,
        d_out=n_cls, n_sweeps=2,
        reflux_ratio=0.3, reboil_ratio=0.2,
    )
    hist_d = train_model(
        model_d, train_ld, test_ld, n_epochs=n_epochs,
        device=device, task=task, verbose=True, print_every=50
    )
    model_d.to('cpu')

    metric = 'accuracy' if task == 'classification' else 'rmse'
    print(f'\nCFNN-A params: {model_a.count_parameters():,} | Final {metric}: {hist_a["test_metrics"][-1]:.4f}')
    print(f'CFNN-D params: {model_d.count_parameters():,} | Final {metric}: {hist_d["test_metrics"][-1]:.4f}')

    return {
        'name': dataset_name, 'task': task,
        'model_a': model_a, 'model_d': model_d,
        'hist_a': hist_a, 'hist_d': hist_d,
        'X_scaled': X_scaled, 'd_in': d_in, 'n_cls': n_cls,
    }

## 3. Dataset 1 — Moons (2D baseline)

Two interleaving half-circles. The simplest test: if CFNN can't solve this,
something is fundamentally broken. With only 2 dimensions, we can fully
interpret the McCabe-Thiele diagram.

In [ ]:
X_moons, y_moons = load_synthetic_dataset('moons', n_samples=1000, noise=0.2, seed=42)
res_moons = train_both_models(X_moons, y_moons, 'Moons (2D)', task='classification', n_plates=8)

### 3.1 McCabe-Thiele — Moons

In [ ]:
fig = mccabe_thiele_plot(
    res_moons['model_a'], res_moons['X_scaled'],
    title='CFNN-A McCabe-Thiele — Moons', show_steps=True, figsize=(8, 8)
)
plt.show()

fig = mccabe_thiele_plot(
    res_moons['model_d'], res_moons['X_scaled'],
    title='CFNN-D McCabe-Thiele — Moons', show_steps=True, figsize=(8, 8)
)
plt.show()

### 3.2 Concentration Profiles — Moons

In [ ]:
fig = concentration_profile(res_moons['model_a'], res_moons['X_scaled'],
                            title='CFNN-A Concentration — Moons')
plt.show()

fig = concentration_profile(res_moons['model_d'], res_moons['X_scaled'],
                            title='CFNN-D Concentration — Moons')
plt.show()

### 3.3 Diagnostic Dashboard — Moons

In [ ]:
fig = diagnostic_dashboard(res_moons['model_a'], res_moons['X_scaled'], model_name='CFNN-A — Moons')
plt.show()

fig = diagnostic_dashboard(res_moons['model_d'], res_moons['X_scaled'], model_name='CFNN-D — Moons')
plt.show()

### 3.4 ChemE Diagnostics — Moons

In [ ]:
print('=== CFNN-A Diagnostics (Moons) ===')
print_diagnostics(res_moons['model_a'], res_moons['X_scaled'])
print('\n=== CFNN-D Diagnostics (Moons) ===')
print_diagnostics(res_moons['model_d'], res_moons['X_scaled'])

---
## 4. Dataset 2 — Wine (13D, correlated chemical features)

This is the **natural domain** of CFNN. Wine has 13 physicochemical
measurements (alcohol, malic acid, ash, alkalinity, magnesium, phenols,
flavanoids, etc.) that interact and correlate — exactly the kind of
continuous, multi-component flow that absorption/distillation towers
process in chemical engineering.

**Key question:** Does the column use more plates effectively when
processing 13 correlated features vs 2 simple features?

In [ ]:
X_wine, y_wine, _ = load_uci_dataset('wine')
res_wine = train_both_models(X_wine, y_wine, 'Wine (13D)', task='classification', n_plates=8)

### 4.1 McCabe-Thiele — Wine

In [ ]:
fig = mccabe_thiele_plot(
    res_wine['model_a'], res_wine['X_scaled'],
    title='CFNN-A McCabe-Thiele — Wine (13 chemical features)', show_steps=True, figsize=(8, 8)
)
plt.show()

fig = mccabe_thiele_plot(
    res_wine['model_d'], res_wine['X_scaled'],
    title='CFNN-D McCabe-Thiele — Wine', show_steps=True, figsize=(8, 8)
)
plt.show()

### 4.2 Transfer Heatmap — Wine

With 13 dimensions, the heatmap becomes especially interesting:
which chemical features does the network transfer at which plates?

In [ ]:
fig = transfer_heatmap(res_wine['model_a'], res_wine['X_scaled'],
                       title='CFNN-A Transfer Heatmap — Wine: which features transfer where?')
plt.show()

fig = transfer_heatmap(res_wine['model_d'], res_wine['X_scaled'],
                       title='CFNN-D Transfer Heatmap — Wine (rect | strip)')
plt.show()

### 4.3 Concentration Profiles — Wine

In [ ]:
fig = concentration_profile(res_wine['model_a'], res_wine['X_scaled'],
                            title='CFNN-A Concentration — Wine')
plt.show()

fig = concentration_profile(res_wine['model_d'], res_wine['X_scaled'],
                            title='CFNN-D Concentration — Wine')
plt.show()

### 4.4 Diagnostic Dashboard — Wine

In [ ]:
fig = diagnostic_dashboard(res_wine['model_a'], res_wine['X_scaled'], model_name='CFNN-A — Wine')
plt.show()

fig = diagnostic_dashboard(res_wine['model_d'], res_wine['X_scaled'], model_name='CFNN-D — Wine')
plt.show()

### 4.5 ChemE Diagnostics — Wine

In [ ]:
print('=== CFNN-A Diagnostics (Wine) ===')
print_diagnostics(res_wine['model_a'], res_wine['X_scaled'])
print('\n=== CFNN-D Diagnostics (Wine) ===')
print_diagnostics(res_wine['model_d'], res_wine['X_scaled'])

---
## 5. Dataset 3 — California Housing (8D, regression)

Now we switch to **regression** with a real-world dataset (20K+ samples).
This tests:
1. That CFNN handles regression (not just classification)
2. Scalability to larger datasets
3. How the column behaves when optimizing a continuous target

**Key question:** In regression, does the McCabe-Thiele diagram look
qualitatively different from classification?

In [ ]:
X_cal, y_cal, _ = load_uci_dataset('california_housing')
res_cal = train_both_models(
    X_cal, y_cal, 'California Housing (8D)',
    task='regression', n_plates=8, n_epochs=100,
)

### 5.1 McCabe-Thiele — California Housing

In [ ]:
fig = mccabe_thiele_plot(
    res_cal['model_a'], res_cal['X_scaled'],
    title='CFNN-A McCabe-Thiele — California Housing (regression)',
    show_steps=True, figsize=(8, 8)
)
plt.show()

fig = mccabe_thiele_plot(
    res_cal['model_d'], res_cal['X_scaled'],
    title='CFNN-D McCabe-Thiele — California Housing',
    show_steps=True, figsize=(8, 8)
)
plt.show()

### 5.2 Driving Force Profiles — California Housing

In regression the driving force distribution may differ significantly
from classification — let's see if the column concentrates effort
in specific plates.

In [ ]:
fig = driving_force_profile(res_cal['model_a'], res_cal['X_scaled'],
                            title='CFNN-A Driving Force — Calif. Housing')
plt.show()

fig = driving_force_profile(res_cal['model_d'], res_cal['X_scaled'],
                            title='CFNN-D Driving Force — Calif. Housing')
plt.show()

### 5.3 Transfer Heatmap — California Housing

In [ ]:
fig = transfer_heatmap(res_cal['model_a'], res_cal['X_scaled'],
                       title='CFNN-A Transfer Heatmap — Calif. Housing')
plt.show()

fig = transfer_heatmap(res_cal['model_d'], res_cal['X_scaled'],
                       title='CFNN-D Transfer Heatmap — Calif. Housing')
plt.show()

### 5.4 Diagnostic Dashboard — California Housing

In [ ]:
fig = diagnostic_dashboard(res_cal['model_a'], res_cal['X_scaled'],
                           model_name='CFNN-A — Calif. Housing')
plt.show()

fig = diagnostic_dashboard(res_cal['model_d'], res_cal['X_scaled'],
                           model_name='CFNN-D — Calif. Housing')
plt.show()

### 5.5 Column Schematics — California Housing

In [ ]:
fig = column_schematic(res_cal['model_a'], res_cal['X_scaled'],
                       title='CFNN-A Column — Calif. Housing')
plt.show()

fig = column_schematic(res_cal['model_d'], res_cal['X_scaled'],
                       title='CFNN-D Column — Calif. Housing')
plt.show()

---
## 6. Cross-Dataset Comparison

The real insight: how does the **same architecture** adapt its internal
behavior to datasets of different complexity?

In [ ]:
# Compare McCabe-Thiele across all 3 datasets (CFNN-A)
from src.diagnostics import operating_line_data

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

datasets = [
    (res_moons, 'Moons (2D)'),
    (res_wine, 'Wine (13D)'),
    (res_cal, 'Calif. Housing (8D)'),
]

for i, (res, label) in enumerate(datasets):
    ax = axes[i]
    op = operating_line_data(res['model_a'], res['X_scaled'])
    gas_n = op.get('gas_norms', op.get('gas_rect_norms', []))
    liq_n = op.get('liquid_norms', op.get('liquid_rect_norms', []))
    ml = min(len(gas_n), len(liq_n))

    ax.plot(liq_n[:ml], gas_n[:ml], 'o-', color='#E74C3C',
            linewidth=2.5, markersize=10)
    for j in range(ml):
        ax.annotate(f'P{j}', (liq_n[j], gas_n[j]),
                    textcoords='offset points', xytext=(8, 8), fontsize=9)

    max_val = max(max(gas_n[:ml]), max(liq_n[:ml])) * 1.1
    ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3)
    ax.set_xlabel('||liquid||', fontsize=12)
    ax.set_ylabel('||gas||', fontsize=12)
    ax.set_title(f'CFNN-A: {label}', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.2)
    ax.set_aspect('equal')

plt.suptitle('McCabe-Thiele Comparison — How the Column Adapts to Data Complexity',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Compare concentration profiles (CFNN-A)
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

for i, (res, label) in enumerate(datasets):
    ax = axes[i]
    result = res['model_a'].forward_with_intermediates(res['X_scaled'])
    gas_norms = [g.norm(dim=-1).mean().item() for g in result['gas_states']]
    liq_norms = [l.norm(dim=-1).mean().item() for l in result['liquid_states']]

    ax.plot(range(len(gas_norms)), gas_norms, 'o-', color='#E74C3C',
            linewidth=2, markersize=8, label='Gas')
    ax.plot(range(len(liq_norms)), liq_norms, 's-', color='#3498DB',
            linewidth=2, markersize=8, label='Liquid')
    ax.fill_between(range(min(len(gas_norms), len(liq_norms))),
                    gas_norms[:min(len(gas_norms), len(liq_norms))],
                    liq_norms[:min(len(gas_norms), len(liq_norms))],
                    alpha=0.1, color='green')
    ax.set_xlabel('Plate', fontsize=12)
    ax.set_ylabel('||stream||', fontsize=12)
    ax.set_title(f'{label}', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.2)

plt.suptitle('Concentration Profiles — Gas & Liquid Norms Through the Tower',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Compare CFNN-A vs CFNN-D for each dataset
fig, axes = plt.subplots(2, 3, figsize=(21, 12))

for i, (res, label) in enumerate(datasets):
    for row, (model_key, arch_name, color) in enumerate([
        ('model_a', 'CFNN-A', '#E74C3C'),
        ('model_d', 'CFNN-D', '#E67E22'),
    ]):
        ax = axes[row, i]
        model = res[model_key]
        result = model.forward_with_intermediates(res['X_scaled'])
        gas_norms = [g.norm(dim=-1).mean().item() for g in result['gas_states']]
        liq_norms = [l.norm(dim=-1).mean().item() for l in result['liquid_states']]
        ml = min(len(gas_norms), len(liq_norms))

        ax.plot(liq_norms[:ml], gas_norms[:ml], 'o-', color=color,
                linewidth=2.5, markersize=9)
        max_val = max(max(gas_norms[:ml]), max(liq_norms[:ml])) * 1.1
        ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3)
        for j in range(ml):
            ax.annotate(f'P{j}', (liq_norms[j], gas_norms[j]),
                        textcoords='offset points', xytext=(6, 6), fontsize=8)
        ax.set_xlabel('||liquid||')
        ax.set_ylabel('||gas||')
        ax.set_title(f'{arch_name} — {label}', fontweight='bold')
        ax.grid(True, alpha=0.2)

plt.suptitle('CFNN-A vs CFNN-D: McCabe-Thiele Across Datasets',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6.1 Summary Table

In [ ]:
import pandas as pd
from src.diagnostics import damkohler_number, number_of_transfer_units, alpha_statistics

rows = []
for res, label in datasets:
    for arch, mk in [('model_a', 'CFNN-A'), ('model_d', 'CFNN-D')]:
        model = res[arch]
        metric_name = 'accuracy' if res['task'] == 'classification' else 'rmse'
        metric_val = res[f'hist_{arch[-1]}']['test_metrics'][-1]
        params = model.count_parameters()

        da = damkohler_number(model, res['X_scaled'])
        ntu = number_of_transfer_units(model, res['X_scaled'])
        alphas = alpha_statistics(model)

        rows.append({
            'Dataset': label,
            'Architecture': mk,
            'Params': params,
            f'Final {metric_name}': f'{metric_val:.4f}',
            'Mean Da': f'{np.mean(da["per_plate"]):.4f}',
            'NTU': f'{ntu["ntu"]:.4f}',
            'Mean alpha': f'{alphas["mean_alpha"]:.4f}',
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

---
## 7. Mathematical Foundations

### 7.1 From Mass Transfer to Neural Networks

In a packed absorption tower, the gas-phase mass balance is:

$$G_s \frac{dY}{dZ} = K_Y a (Y - Y^*)$$

where $Y$ is the gas mole ratio, $Y^* = f(X)$ is the equilibrium value, and $K_Y a$ is the overall mass transfer coefficient.

**CFNN translates this to:**

$$\delta_n = g_n - E(l_n) \quad \text{(driving force, analog of } Y - Y^* \text{)}$$

$$\Delta_n = \alpha \cdot \tanh(T(\delta_n)) \quad \text{(transfer, analog of } K_Y a \cdot (Y - Y^*) \cdot \Delta Z \text{)}$$

$$g_{n+1} = g_n - \Delta_n, \quad l_n = l_{n+1} + \Delta_n \quad \text{(conservation)}$$

### 7.2 Conservation Law

The fundamental constraint:

$$\Delta g = -\Delta l$$

This is a **hard constraint**, not a soft regularization. What the gas stream loses at each plate, the liquid stream gains. This is equivalent to conservation of mass in a real absorption tower.

### 7.3 Equilibrium Function

The learned equilibrium $E(l) = \sigma(Wl + b)$ maps the liquid state to the "expected" gas state. In ChemE terms, this is the VLE (vapor-liquid equilibrium) curve $Y^* = f(X)$.

The sigmoid ensures $E(l) \in (0, 1)$, bounding the equilibrium estimate. This is analogous to physical systems where compositions are bounded between 0 and 1.

### 7.4 Approach C: Unrolled Bidirectional

Real absorption towers have a chicken-and-egg problem: to compute liquid at plate $n$, you need gas at plate $n-1$, but gas depends on liquid from above.

**Approach C** resolves this by alternating sweeps:
1. **Sweep down**: liquid descends using current gas estimates
2. **Sweep up**: gas ascends using freshly computed liquid values
3. Repeat for $S$ sweeps (typically $S = 2$)

This is analogous to iterative tray-by-tray calculations until convergence.

### 7.5 CFNN-D: Distillation Extension

Distillation adds:
- **Bidirectional transfer**: both $\alpha$ (condensation, gas$\to$liquid) and $\beta$ (vaporization, liquid$\to$gas)
- **Feed plate**: input enters at a specific location, splitting the column into rectifying (above) and stripping (below) sections
- **q-line**: learned feed condition $q \in (0,1)$ determines the gas/liquid split of the feed
- **Reflux**: fraction $R$ of top product recycled as liquid (condenser)
- **Reboil**: fraction of bottom product recycled as gas (reboiler)

The McCabe-Thiele diagram shows two operating lines (rectifying and stripping) with different slopes, intersecting at the feed plate.

### 7.6 Diagnostic Metrics

| Metric | ChemE Meaning | CFNN Analog |
|--------|--------------|-------------|
| **Damkohler number** (Da) | Reaction rate / flow rate | Transfer rate / information flow rate |
| **Murphree efficiency** ($\eta$) | Actual change / equilibrium change | How close each plate gets to equilibrium |
| **NTU** | $\int dY/(Y-Y^*)$ | Effective network depth |
| **Reflux ratio** (R) | L/D at condenser | Fraction of top output recycled |
| **Feed condition** (q) | Liquid fraction of feed | Learned gas/liquid split of input |

---
## Phase 3 Complete!

### Key Observations

| Observation | Moons (2D) | Wine (13D) | Calif. Housing (8D) |
|-------------|-----------|------------|--------------------|
| McCabe-Thiele shape | Simple curve | Complex trajectory | Regression pattern |
| Active plates | Few needed | More engaged | Task-dependent |
| Transfer heatmap | 2 dims only | Rich 13-dim pattern | 8-dim structure |
| CFNN-A vs CFNN-D | Similar | D may leverage sections | D adds flexibility |

**Key functions** (importable from `src.visualization`):
- `mccabe_thiele_plot()` — the signature CFNN diagram
- `concentration_profile()` — feature norms through the tower
- `driving_force_profile()` — driving force and transfer at each plate
- `transfer_heatmap()` — per-dimension exchange patterns
- `diagnostic_dashboard()` — complete 6-panel analysis
- `column_schematic()` — visual tower diagram

**Next steps (Phase 4):**
- Build Gradio Space (`app.py`) with interactive demo
- Deploy to HuggingFace Hub